# 02 — `alexandria-server`: Governance API (`/v1`), VC 2.0 Credentials & Immutable `/pub`

This notebook walks through **Part III (`C-HUB`)** of Specification v0.3.0:
1. **OIDC Discovery & `at+jwt` Authentication (`A1`–`A5`)**: `deploy/fake-idp` serving `/.well-known/openid-configuration` and ES256 JWKS, minting tokens for `ana` (`vhub-admin`), `sergio` (namespace steward), and `eve` (unprivileged).
2. **Namespaces & Stewardship**: `POST /v1/namespaces`, `POST/GET/DELETE /v1/namespaces/{ns}/stewards`, plus `401 API_UNAUTHENTICATED` and `403 API_FORBIDDEN_ROLE` checks.
3. **Managed Vocabularies**: Both `proxy` vocabularies (`Link: <upstream>; rel="via"` + `:refresh`) and `hosted` vocabularies (`PUT .../content` with `ETag` / `If-Match`).
4. **Authoring & Optimistic Concurrency**: Draft CRUD, `428 API_PRECONDITION_REQUIRED`, `412 API_PRECONDITION_FAILED`, `:validate`, `:propose`, `:reject`, and `:approve`.
5. **W3C VC 2.0 & Immutable `/pub`**: Inspecting `VersionListing`, `VersionRecord`, and `PackageManifest` signed with `eddsa-jcs-2022` (`did:key:z6Mk...`), `/pub/_vhub/v1/*`, `/pub/demo` `303` redirects, `/pub/packages/{segment}/manifest.json`, and `/pub/blobs/sha256/{hex}`.

In [ ]:
from lab import AlexandriaLab

lab = AlexandriaLab(namespace="demo", compiler="poc", initial_clock="2026-09-01T09:00:00Z")
lab.start_hub_stack()

ana = lab.token("ana", admin=True)
sergio = lab.token("sergio", admin=False)
eve = lab.token("eve", admin=False)

## 1. OIDC Discovery, JWKS & Role-Based Access Control

`alexandria-server` validates incoming `Authorization: Bearer <at+jwt>` tokens against the configured OIDC issuer's JWKS (`jwt_jwks`). Public read endpoints require no token, hub-wide mutations require `vhub-admin` (`ana`), and namespace authoring requires stewardship (`sergio`).

In [ ]:
st, _, oidc = lab.request("GET", f"{lab.idp_url}/.well-known/openid-configuration")
assert st == 200 and oidc["issuer"] == lab.idp_url

# Missing token -> 401 API_UNAUTHENTICATED with WWW-Authenticate: Bearer
st, hdr, err401 = lab.request("POST", f"{lab.hub_url}/v1/namespaces", body={"slug": "demo"})
assert st == 401 and err401["code"] == "API_UNAUTHENTICATED"

# Non-admin token -> 403 API_FORBIDDEN_ROLE
st, _, err403 = lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=sergio, body={"slug": "demo"})
assert st == 403 and err403["code"] == "API_FORBIDDEN_ROLE"

# Admin creates namespace 'demo' and assigns 'sergio' + temporary steward 'temp-user'
st, _, ns_doc = lab.request("POST", f"{lab.hub_url}/v1/namespaces", token=ana, body={"slug": "demo"})
assert st == 201

st, _, stw_sergio = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/stewards", token=ana, body={"iss": lab.idp_url, "sub": "sergio"})
assert st == 201
st, _, stw_temp = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/stewards", token=ana, body={"iss": lab.idp_url, "sub": "temp-user"})
assert st == 201
temp_id = stw_temp["id"].rsplit("/", 1)[-1]

st, _, stw_list = lab.request("GET", f"{lab.hub_url}/v1/namespaces/demo/stewards", token=sergio)
assert st == 200 and len(stw_list["@graph"]) == 2

st, _, _ = lab.request("DELETE", f"{lab.hub_url}/v1/namespaces/demo/stewards/{temp_id}", token=ana)
assert st == 204

## 2. Proxy & Hosted Vocabularies (`/v1/namespaces/demo/vocabularies` & `/pub/demo/vocab/...`)

Stewards can register two vocabulary kinds:
- **`proxy`**: Fetches and snapshots an upstream semantic resource into immutable revisions (`r/1`, `r/2`, ...), serving `Link: <upstream>; rel="via"` on `/pub`.
- **`hosted`**: Stores steward-uploaded Turtle, JSON-LD, or JSON Schema directly with `ETag` / `If-Match` concurrency control.

In [ ]:
# 1. Proxy vocabulary backed by fake-connector's /dcat-ap.jsonld
st, _, voc_proxy = lab.request(
    "POST", f"{lab.hub_url}/v1/namespaces/demo/vocabularies",
    token=sergio,
    body={
        "name": "dcat-ap-context",
        "kind": "proxy",
        "media_type": "application/ld+json",
        "upstream": f"{lab.conn_url}/dcat-ap.jsonld",
    },
)
assert st == 201 and len(voc_proxy["revisions"]) == 1

# Refreshing when upstream bytes have not changed returns 200 OK (idempotent revision r/1)
st, _, _ = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/vocabularies/dcat-ap-context:refresh", token=sergio)
assert st == 200

# Public dereference: 303 See Other -> immutable /r/1 with Link: rel="via"
st, hdr303, _ = lab.request("GET", f"{lab.hub_url}/pub/demo/vocab/dcat-ap-context", follow_redirects=False)
assert st == 303 and hdr303["location"].endswith("/pub/demo/vocab/dcat-ap-context/r/1")

st, hdr_r1, _ = lab.request("GET", f"{lab.hub_url}/pub/demo/vocab/dcat-ap-context/r/1")
assert st == 200 and 'rel="via"' in hdr_r1.get("link", "") and "immutable" in hdr_r1.get("cache-control", "")

# 2. Hosted Turtle vocabulary
st, _, _ = lab.request(
    "POST", f"{lab.hub_url}/v1/namespaces/demo/vocabularies",
    token=sergio,
    body={"name": "terms", "kind": "hosted", "media_type": "text/turtle"},
)
assert st == 201

ttl_v1 = "@prefix ex: <https://example.org/ns#> .\nex:access a <http://www.w3.org/2002/07/owl#ObjectProperty> .\n"
st, hdr_put, _ = lab.request(
    "PUT", f"{lab.hub_url}/v1/namespaces/demo/vocabularies/terms/content",
    token=sergio, headers={"Content-Type": "text/turtle"}, body=ttl_v1,
)
assert st == 201 and "etag" in hdr_put

## 3. Draft Authoring, `If-Match` Concurrency, Candidate Rejection & Approval

Let's test the full authoring workflow on `/v1/namespaces/demo/drafts`:
1. Unprivileged principal `eve` is rejected with `403 API_FORBIDDEN_ROLE`.
2. Steward `sergio` creates a draft (`ETag: "1"`), verifies `428` without `If-Match` and `412` with a stale `ETag`, updates with `If-Match: "1"` (`ETag: "2"`), validates (`:validate`), proposes (`:propose`), rejects (`:reject` returning the candidate to draft), re-proposes, and approves `1.0.0` (`:approve`).

In [ ]:
lab.set_clock("2026-09-01T10:00:00Z")

draft_100 = {
    "version": "1.0.0",
    "profiles": [f"{lab.hub_url}/pub/demo/profiles/odrl/1.0.0"],
    "slots": [{"slot": "policy", "obligation": "mandatory", "applies_to": ["policy_intake"]}],
    "files": [
        {
            "path": "schemas/policy.schema.json",
            "media_type": "application/schema+json",
            "role": "json-schema",
            "slot": "policy",
            "applies_to": ["policy_intake"],
            "content": {
                "type": "object",
                "properties": {"action": {"enum": ["odrl:use", "ex:access"]}},
                "required": ["action"],
            },
        }
    ],
}

# Eve cannot create drafts in 'demo'
st, _, err_eve = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts", token=eve, body=draft_100)
assert st == 403 and err_eve["code"] == "API_FORBIDDEN_ROLE"

# Sergio creates draft
st, hdr_d1, d1 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts", token=sergio, body=draft_100)
assert st == 201
draft_id = d1["id"].rsplit("/", 1)[-1]
etag_1 = hdr_d1["etag"]

# Missing If-Match -> 428; wrong If-Match -> 412; valid If-Match -> 200
st, _, e428 = lab.request("PUT", f"{lab.hub_url}/v1/namespaces/demo/drafts/{draft_id}", token=sergio, body=draft_100)
assert st == 428 and e428["code"] == "API_PRECONDITION_REQUIRED"

st, _, e412 = lab.request("PUT", f"{lab.hub_url}/v1/namespaces/demo/drafts/{draft_id}", token=sergio, headers={"If-Match": '"999"'}, body=draft_100)
assert st == 412 and e412["code"] == "API_PRECONDITION_FAILED"

st, hdr_d2, _ = lab.request("PUT", f"{lab.hub_url}/v1/namespaces/demo/drafts/{draft_id}", token=sergio, headers={"If-Match": etag_1}, body=draft_100)
assert st == 200

# Validate -> Propose -> Reject -> Re-propose -> Approve
st, _, vrep = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{draft_id}:validate", token=sergio)
assert st == 200 and vrep["valid"] is True and vrep["class"] == "initial"

st, _, c1 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{draft_id}:propose", token=sergio)
assert st == 201
cand_1 = c1["id"].rsplit("/", 1)[-1]

st, _, c1_rej = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{cand_1}:reject", token=sergio, body={"comment": "Verify wording before initial release"})
assert st == 200 and c1_rej["state"] == "rejected"

st, _, c2 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/drafts/{draft_id}:propose", token=sergio)
assert st == 201
cand_2 = c2["id"].rsplit("/", 1)[-1]

st, _, rec_100 = lab.request("POST", f"{lab.hub_url}/v1/namespaces/demo/candidates/{cand_2}:approve", token=sergio)
assert st == 201 and "VersionRecord" in rec_100["type"]

## 4. W3C VC 2.0 Credentials, Content-Addressed `/pub` & Audit Events

Approving `1.0.0` atomically:
1. Writes content-addressed blobs under `/pub/blobs/sha256/{hex}`.
2. Signs and publishes `PackageManifest` at `/pub/packages/{base64url-ni-digest}/manifest.json` (`Cache-Control: public, max-age=31536000, immutable`).
3. Signs `VersionRecord` and issues a fresh `VersionListing` (`GET /v1/namespaces/demo/versions`).
4. Activates `/pub/demo` (`303 See Other`) once the clock reaches `new_content_from` (`published_at + propagation_window`).

In [ ]:
try:
    # 1. Inspect signed VersionListing VC
    st, _, listing_vc = lab.request("GET", f"{lab.hub_url}/v1/namespaces/demo/versions")
    assert st == 200 and "VersionListing" in listing_vc["type"]
    assert listing_vc["issuer"] == lab.hub_did

    manifest_ni = listing_vc["credentialSubject"]["versions"][0]["manifest"]
    manifest_seg = manifest_ni.removeprefix("ni:///sha-256;")

    # 2. Fetch immutable signed PackageManifest VC & content-addressed blob
    st, hdr_mf, manifest_vc = lab.request("GET", f"{lab.hub_url}/pub/packages/{manifest_seg}/manifest.json")
    assert st == 200 and "PackageManifest" in manifest_vc["type"] and "immutable" in hdr_mf.get("cache-control", "")

    import base64
    file_entry = manifest_vc["credentialSubject"]["files"][0]
    b64 = file_entry["digest"].removeprefix("ni:///sha-256;")
    blob_hex = base64.urlsafe_b64decode(b64 + "=" * (-len(b64) % 4)).hex()

    st, _, blob_doc = lab.request("GET", f"{lab.hub_url}/pub/blobs/sha256/{blob_hex}")
    assert st == 200 and blob_doc["required"] == ["action"]

    # 3. Before new_content_from (2026-09-03T10:00:00Z), /pub/demo is Scheduled -> 404; after -> 303 See Other
    st_sched, _, _ = lab.request("GET", f"{lab.hub_url}/pub/demo", follow_redirects=False)
    assert st_sched == 404

    lab.set_clock("2026-09-03T10:05:00Z")
    st_live, hdr_live, _ = lab.request("GET", f"{lab.hub_url}/pub/demo", follow_redirects=False)
    assert st_live == 303 and "location" in hdr_live

    # 4. Self-hosted vhub JSON-LD context & vocabulary, plus namespace audit events
    st, _, _ = lab.request("GET", f"{lab.hub_url}/pub/_vhub/v1/context.jsonld")
    assert st == 200
    st, _, _ = lab.request("GET", f"{lab.hub_url}/pub/_vhub/v1/vocab.ttl")
    assert st == 200

    st, _, evs = lab.request("GET", f"{lab.hub_url}/v1/namespaces/demo/events?limit=20", token=sergio)
    assert st == 200 and len(evs["@graph"]) >= 5
finally:
    lab.close()